# Tarea puente, guía L2: el índice sancocho

Cálculo Diferencial, laboratorio computacional. Universidad Nacional de Colombia, Sede Manizales.

Responsable: Thomas Ramírez Villa. Equipo: Nicol Velez Arboleda, Sofía Hoyos Tabares, Sebastián Gutiérrez Rodríguez y Thomas Ramírez Villa.


In [ ]:
import pandas as pd

# Congelado del curso (el mismo que usan los cuadernos de las sesiones)
RUTA = "https://raw.githubusercontent.com/cmorregof/labs-calculo-2026-2/main/datos/sipsa_mensual.csv.gz"
df = pd.read_csv(RUTA)
print(df.shape)

## Verificación del congelado

La celda de abajo clona el repositorio del curso y reconstruye el congelado desde los ocho ZIP oficiales del DANE (uno para 2013 a 2017 y uno por año de 2018 a 2024). Después lo compara con `datos/README.md`: 666.865 filas, 144 meses (2013-01 a 2024-12), 71 ciudades, 385 productos y el plátano dominico hartón verde de Manizales, Centro Galerías, a \$2.663/kg en 2024-12.

Relato de la diferencia (5 de noviembre de 2026). El clon funcionó, pero `congelar_sipsa.py` se detuvo en la primera descarga: el portal de microdatos del DANE respondió `HTTP Error 401: Unauthorized`, porque sin una sesión iniciada no deja bajar los ZIP. La celda revisa entonces los mismos controles sobre el congelado del curso, y los seis coinciden. Eso muestra que el congelado cumple lo que dice el README. La reproducción desde la fuente queda pendiente: habría que iniciar sesión en el portal del DANE, bajar los ocho ZIP a mano y adaptar el script para que los lea desde el disco.


In [24]:
import os, subprocess
VERIFICAR_CONGELADO = True   # descarga ~8 ZIP del DANE; si el DANE está caído, el cuaderno sigue corriendo
try:
    if not os.path.exists("labs-calculo-2026-2"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/cmorregof/labs-calculo-2026-2"], check=True)
    if VERIFICAR_CONGELADO and not os.path.exists("/tmp/sipsa_mensual.csv.gz"):
        r = subprocess.run(["python3", "datos/congelar_sipsa.py", "/tmp/sipsa_mensual.csv.gz"],
                           cwd="labs-calculo-2026-2", capture_output=True, text=True)
        print(r.stdout[-1500:])
        if r.returncode != 0:
            raise RuntimeError(r.stderr.strip().splitlines()[-1] if r.stderr.strip() else f"código {r.returncode}")
    nuevo = pd.read_csv("/tmp/sipsa_mensual.csv.gz")
    reconstruido = True
except Exception as e:
    print("No se pudo reconstruir el congelado desde el DANE:", e)
    print("Se revisan los mismos controles sobre el congelado del curso (df). Esto lo verifica, pero no lo reproduce;")
    print("para reproducirlo hace falta acceso al portal de microdatos del DANE.")
    nuevo = df.copy()
    reconstruido = False

control = nuevo[(nuevo["producto"] == "Plátano dominico hartón verde")
                & (nuevo["mercado"] == "Manizales, Centro Galerías") & (nuevo["fecha"] == "2024-12")]["precio_kg"]
chequeos = {
    "filas = 666 865":         len(nuevo) == 666_865,
    "meses = 144":             nuevo["fecha"].nunique() == 144,
    "2013-01 a 2024-12":       (nuevo["fecha"].min(), nuevo["fecha"].max()) == ("2013-01", "2024-12"),
    "ciudades = 71":           nuevo["ciudad"].nunique() == 71,
    "productos = 385":         nuevo["producto"].nunique() == 385,
    "plátano MZL 2024-12 = 2663": len(control) == 1 and round(control.iloc[0]) == 2663,
}
for nombre, ok in chequeos.items():
    print("[OK]   " if ok else "[FALLA]", nombre)
print(f"\nFilas: {len(nuevo):,} · meses: {nuevo['fecha'].nunique()} · ciudades: {nuevo['ciudad'].nunique()} · "
      f"productos: {nuevo['producto'].nunique()} · plátano: ${control.iloc[0]:,.0f}")
iguales = nuevo.sort_values(list(nuevo.columns[:4])).reset_index(drop=True).equals(
          df.sort_values(list(df.columns[:4])).reset_index(drop=True))
print("¿El congelado reconstruido es idéntico al del curso?", iguales if reconstruido else "no aplica (no se reconstruyó)")
# Pantallazo de esta salida para la entrega.

Descargando 2013-2017…

No se pudo reconstruir el congelado desde el DANE: urllib.error.HTTPError: HTTP Error 401: Unauthorized
Se revisan los mismos controles sobre el congelado del curso (df). Esto lo verifica, pero no lo reproduce;
para reproducirlo hace falta acceso al portal de microdatos del DANE.
[OK]    filas = 666 865
[OK]    meses = 144
[OK]    2013-01 a 2024-12
[OK]    ciudades = 71
[OK]    productos = 385
[OK]    plátano MZL 2024-12 = 2663

Filas: 666,865 · meses: 144 · ciudades: 71 · productos: 385 · plátano: $2,663
¿El congelado reconstruido es idéntico al del curso? no aplica (no se reconstruyó)


Defensa del precio medio (Thomas)

1. Sirve para seguir la tendencia del costo de la comida, porque se reporta cada mes, no salta con un solo vendedor y deja comparar años y ciudades. Como medida del costo de vida de una familia se queda corta: es el precio mayorista de una central, y en la tienda se paga más.
2. Pagan más que ese promedio quienes compran al detal y en poca cantidad (tienda de barrio, minimercado), quienes viven lejos de la central y cargan con transporte e intermediarios, y quienes no pueden esperar a que el precio baje ni guardar comida.
3. La familia además gasta en lo que el índice no cuenta: cilantro, cebolla, ajo, gas, arriendo y transporte.
4. Para un índice del precio máximo o de la mediana harían falta las cotizaciones individuales de cada día y de cada vendedor (los microdatos de SIPSA). El congelado solo trae el promedio mensual.
5. Para un índice minorista harían falta precios de tienda y supermercado por barrio y ciudad, el tamaño de los empaques y las cantidades que compra cada hogar.
